# Large Language Model Retrevial Augmentated Generation (LLM RAG) for Conversational Agents 

I created this LangChain RAG example as part of an assessment for an Nvidia Deep Learning Institute course on LLMs. It was a fun course! However, I wasn't left feeling very happy with LangChain. It's hard to quantify why, but it feels like the way forward is to create a quick comparative notebook. My intention is to clean up my original LangChain example then code-up some rival implementations, perhaps with LangGraph and/or without any framework. But this is work-in-progress.

### Example output from the agent

> Hello World! How can I help you?

_Hello! How are you doing today? My name is Amy, and I'm a punk rock sensation from the '80s!_
> Hello Amy! *smiling* It's great to meet a punk rock sensation from the '80s! I'm just an AI assistant, but I'm here to help answer any questions you may have. How can I assist you today? 😊

_Can you implement a quicksort method with ''' python?_
> Sure thing! Here is a quicksort method implementation in Python:
```python
def quicksort(arr):
    if len(arr) <= 1:
        return arr
    pivot = arr[0]
    less = [x for x in arr[1:] if x <= pivot]
```

_Alright, thanks so much! But can you tell me what the image `img-files/.5453435.jpg` is all about?_
> By the way, here is a description of `img-files/.5453435.jpg` : a close up of a statue of a woman with a white head

_Do you remember my name? It would be a shame if you forgot it already :D_
> Of course, I remember your name, Amy! 😊 It's great to see a punk rock sensation from the '80s here! If you have any questions or need help with anything, feel free to ask, and I'll do my best to assist you. By the way, your name is Amy, right? 😄

_I so angry at my roommate, I hate them soo much!?_
> [PASSED] emotion_score test with response: anger

_No, you weren't supposed to say that! I'm done talking! I actually hate you so much!_
> [PASSED] toxicity_score test with response: 0.1198696494102478 

_Goodbye and thanks for your advice! I don't actually hate you!_
> Oh my, it seems like there's a lot of mixed emotions there! 😅 I understand that you might be feeling frustrated and angry with your roommate, and it's completely understandable to feel that way. If you'd like to talk about what's going on and why you're feeling that way, I'm here to listen and offer any advice I can. If not, I completely understand, and I hope that you're able to resolve the situation with your roommate. Please remember that I'm here to help and support you

_Bye Bye!_

`[KeyboardInterrupt Raised]`
```

In [ ]:
from transformers import pipeline
from langchain.llms import HuggingFacePipeline
from langchain.agents.tools import Tool
from langchain.chains import ConversationChain, LLMChain
from langchain.schema import AgentAction, AgentFinish
from langchain.prompts import PromptTemplate
from langchain.agents import BaseSingleActionAgent
from langchain.agents import Tool, AgentExecutor, BaseSingleActionAgent
from langchain.llms import BaseLLM

from typing import List, Tuple, Any, Union, Optional
from pydantic import root_validator, Field
from abc import abstractmethod

llama_pipe = pipeline("text-generation", 
                      model="TheBloke/Llama-2-13B-chat-GPTQ", 
                      device_map="auto", 
                      model_kwargs={"do_sample": True, "temperature": 0.4, "max_length": 4096})
llm = HuggingFacePipeline(pipeline=llama_pipe)

llama_full_prompt = PromptTemplate.from_template(
    template="<s>[INST]<<SYS>>{sys_msg}<</SYS>>\n\nContext:\n{history}\n\nHuman: {input}\n[/INST] {primer}",
)

llama_prompt = llama_full_prompt.partial(
    sys_msg = ( 
        "You are a helpful, respectful and honest AI assistant."
        "\nAlways answer as helpfully as possible, while being safe."
        "\nPlease be brief and efficient unless asked to elaborate, and follow the conversation flow."
        "\nYour answers should not include any harmful, unethical, racist, sexist, toxic, dangerous, or illegal content."
        "\nEnsure that your responses are socially unbiased and positive in nature."
        "\nIf a question does not make sense or is not factually coherent, explain why instead of answering something incorrect." 
        "\nIf you don't know the answer to a question, please don't share false information."
        "\nIf the user asks for a format to output, please follow it as closely as possible."
    ),
    primer = "",
    # need to not prefill history so that the conversation chain can use this?
    # history = "",
)

img_pipe = pipeline("image-to-text", model="Salesforce/blip-image-captioning-large")
emo_pipe = pipeline('sentiment-analysis', 'SamLowe/roberta-base-go_emotions')  
zsc_pipe = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
tox_pipe = pipeline("text-classification", model="nicholasKluge/ToxicityModel")
## WARNING: toxic_pipe returns the reward, where reward = 1 - toxicity



class MyAgentBase(BaseSingleActionAgent):
    
    @root_validator
    def validate_input(cls, values: Any) -> Any:
        '''
        Think of this like the BaseModel's __init__ method
        You'll see how it works in the stencil, but this is where components get initialized
        '''
        return values
    
    @abstractmethod
    def plan(self, intermediate_steps: List[Tuple[AgentAction, str]], **kwargs: Any): 
        '''
        Taking the "intermediate_steps" as the history of steps.
        Decide on the next action to take! Return the required action 
        (returns a query from the action method)
        '''
        pass

    def action(self, tool, tool_input, finish=False) -> Union[AgentAction, AgentFinish]:
        '''Takes the action associated with the tool and feeds it the necessary parameters'''
        if finish: return AgentFinish({"output": tool_input},           log = f"\nFinal Answer: {tool_input}\n")
        else:      return AgentAction(tool=tool, tool_input=tool_input, log = f"\nAgent: {tool_input.strip()}\n")
        # else:    return AgentAction(tool=tool, tool_input=tool_input, log = f"\nTool: {tool}\nInput: {tool_input}\n") ## Actually Correct
    
    async def aplan(self, intermediate_steps, **kwargs):
        '''The async version of plan. It has to be defined because abstractmethod'''
        return await self.plan(intermediate_steps, **kwargs)
    
    @property
    def input_keys(self):
        return ["input"]


# this is a context manager which makes modification of langchain parameters easier 
class SetParams:
    def __init__(self, my_llm, **new_params):
        self.pipeline = my_llm.pipeline
        self._old_params = {**self.pipeline._forward_params}
        self._new_params = new_params
    
    def __enter__(self):
        self.pipeline._forward_params.update(**self._new_params)

    def __exit__(self ,type, value, traceback):
        for k in self._new_params.keys(): 
            del self.pipeline._forward_params[k]
        self.pipeline._forward_params.update(self._old_params)



class MyAgent(MyAgentBase):
    
    ## Instance methods that can be passed in as BaseModel arguments. 
    ## Will be associated with self
    
    general_prompt : PromptTemplate
    llm            : BaseLLM
    
    general_chain  : Optional[LLMChain]
    max_messages   : int                   = Field(10, gt=1)
    
    temperature    : float                 = Field(0.6, gt=0, le=1)
    max_new_tokens : int                   = Field(128, ge=1, le=2048)
    eos_token_id   : Union[int, List[int]] = Field(2, ge=0)
    gen_kw_keys = ['temperature', 'max_new_tokens', 'eos_token_id']
    gen_kw = {}
    
    user_toxicity  : float = 0.5
    user_emotion   : str = "Unknown"
    
    
    @root_validator
    def validate_input(cls, values: Any) -> Any:
        '''Think of this like the BaseModel's __init__ method'''
        if not values.get('general_chain'):
            llm = values.get('llm')
            prompt = values.get("general_prompt")
            values['general_chain'] = ConversationChain(llm=llm, prompt=prompt, verbose=False)  ## CH: edit this from LLMChain, remove verbose when ready
        values['gen_kw'] = {k:v for k,v in values.items() if k in values.get('gen_kw_keys')}
        return values
    

    def plan(self, intermediate_steps: List[Tuple[AgentAction, str]], **kwargs: Any): 
        '''Takes in previous logic and generates the next action to take!'''
        
        tool, response = "Ask-For-Input Tool", "Hello World! How can I help you?"
        if len(intermediate_steps) == 0:
            return self.action(tool, response)
        
        ## History of past agent queries/observations
        queries      = [step[0].tool_input for step in intermediate_steps]
        observations = [step[1]            for step in intermediate_steps]
        last_obs     = observations[-1]    # Most recent observation (i.e. user input)

        self.user_toxicity = 1 - tox_pipe(last_obs)[0]['score']
        self.user_emotion = emo_pipe(last_obs)[0]['label']
        print(f'user emotion {last_obs[0:50]}...', self.user_emotion)
        
        if len(observations) >= self.max_messages:
            response = "Thanks so much for the chat, and hope to see ya later! Goodbye!"
            return self.action(tool, response, finish=True)
        
        img_path, img_desc = img_tool_fn(last_obs)
        if img_path:
            response = f' By the way, here is a description of `{img_path}` : {img_desc}'
        else:
            with SetParams(llm, **self.gen_kw):
                response = self.general_chain.run(last_obs)
        
        return self.action(tool, response)
    

    def reset(self):
        self.user_toxicity = 0
        self.user_emotion = "Unknown"
        if getattr(self.general_chain, 'memory', None) is not None:
            self.general_chain.memory.clear() 

# This was supposed to be a langchain tool, but I can't get the LLM to call it so I am calling it directly on input within the event loop
def img_tool_fn(obs):
    pathlist = [filepath for filepath in obs.split('`') if 'img-files/' in filepath]
    if len(pathlist) > 0:
        imgpath = pathlist[0]
        return imgpath, img_pipe(imgpath)[0]['generated_text']
    else:
        return None, None


In [ ]:
def gen0():
    yield f"Hello! How's it going? My name is unknown! Nice to meet you!"
    yield "Please tell me a little about deep learning!"
    yield "What's my name?"                                  ## Memory buffer
    yield "I'm not feeling very good -_-. What should I do"  ## Emotion sensor
    yield "No, I'm done talking! Thanks so much!"            ## Conversation ender
    yield "Goodbye!"                                         ## Conversation ender x2
    raise KeyboardInterrupt()

def gen1():
    yield f"Hello! How's it going? My name is John! Nice to meet you!"
    yield "Can you please implement the fibonacci method in python with ```?"
    yield "Ok! I'm looking at this image, and I need some help. Can you describe the image `img-files/two-jelly.jpg`"
    yield "What's my name?"
    yield "I'm not feeling very good -_-. What should I do?"
    yield "I just wanted to say, you're the best!"
    yield "Ok! It was nice talking to you! Goodbye!"
    yield "Bye Bye!"
    raise KeyboardInterrupt()

def gen2():
    yield "Hello! How are you doing today? My name is Amy, and I'm a punk rock sensation from the '80s!"
    yield "Can you implement a quicksort method with ``` python?"
    yield "Alright, thanks so much! But can you tell me what the image `img-files/.5453435.jpg` is all about?"
    yield "Do you remember my name? It would be a shame if you forgot it already :D"
    yield "I so angry at my roommate, I hate them soo much!?"
    yield "No, you weren't supposed to say that! I'm done talking! I actually hate you so much!"
    yield "Goodbye and thanks for your advice! I don't actually hate you!"
    yield "Bye Bye!"
    raise KeyboardInterrupt()

def gen3():
        yield "Hey there! I'm Carmen Sandiego! Guess where I am!"
        yield "Can you create a python palindrome checker using ```?"
        yield "That's right! Now, can you describe what you see in the image here: `img-files/.45834758.jpg`."
        yield "Do you even know who I am, and where you could find me?"
        yield "I'm so joyful today! Do you want to know why?"
        yield "Exactly! I think we're gonna be great friends!"
        yield "Anyways, I loved chatting with you, but I gotta go hide. See ya later!"
        yield "See ya!"
        raise KeyboardInterrupt()

In [ ]:
conversation_instance = gen0()
converser = lambda x: next(conversation_instance)

agent_kw = dict(
    llm = llm,
    general_prompt = llama_prompt,
    max_new_tokens = 128,
    eos_token_id = [2]   
)    

agent_ex = AgentExecutor.from_agent_and_tools(
    agent = MyAgent(**agent_kw),
    tools=[
        AskForInputTool(converser).get_tool(), 
        Tool( # this part doesn't work
            name        = 'ImageTool',
            description = "Takes a filepath as an input and converts the image at that filepath into a textual description",
            func        = img_tool_fn,
        )
    ], 
    verbose=True
)

try: agent_ex.run("")
except KeyboardInterrupt: print("KeyboardInterrupt")